# Music Annotation Project
## Notebook 76: Ring-2 Whole-Page Human Preference and Objectionability Review

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook answers a visual-restoration question that pixel metrics cannot answer alone:

> When C1 restores extra printed notation in Ring 2 but also restores some annotation, does a human reader prefer the C1 page to the incumbent, and are the mathematically false/catastrophic cases actually visually objectionable enough to justify a future automatic veto?

### Source experiment

Notebook 76 is a **read-only human-review layer over completed Notebook 75 outputs**.

- `C0_INCUMBENT` = frozen incumbent correction.
- `C1_PRESERVE_RING2` = exact observed bytes restored in Ring 2 only.
- C2 is not part of the primary review because Notebook 75/75A already showed that Ring 1+2 is a fundamentally different and much less selective intervention.

### Review philosophy

Preference is not treated as a mathematical safety metric. The review separately records:

1. which page a human would rather perform/read from;
2. whether C1 visibly restores useful printed notation;
3. whether C1 visibly restores annotation;
4. whether the restored annotation interferes with reading the music;
5. whether each version remains playable/readable;
6. whether the reviewer would choose C1 for performance despite any returned annotation.

Ties are allowed. A reviewer may prefer C1 even when some annotation returns, provided the net visual result is better for reading the music.

### Blinding

The initial comparison is randomised A/B per page. The key is sealed until all required reviews are completed and the human-review lock is closed. Clean truth, numerical truth labels and false/catastrophic flags are not shown during the initial judgement.

### After the review lock closes

Only then does the notebook decode A/B and join the human judgements to Notebook 75/75A truth-side evidence. This allows the mathematically false-heavy and catastrophic C1 cases to be inspected as *visual* outcomes without allowing those labels to bias the initial reviewer.

### No production authority

Notebook 76 does not change `v8_8_operator.py`, does not promote Ring 2, and does not invent a new automatic veto rule. It produces evidence for the next design decision.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy matplotlib seaborn gradio pillow

## 1. Configuration and source-run resolution

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, hmac, secrets, shutil, zipfile, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from PIL import Image, ImageOps, ImageDraw

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB75_ROOT=ROOT/'notebook75_v88_collar_preservation_counterfactual'
NB75A_ROOT=ROOT/'notebook75a_stratified_reanalysis'
OUTPUT_ROOT=ROOT/'notebook76_ring2_human_review'
SOURCE_NB75_RUN_OVERRIDE=None
SOURCE_NB75A_RUN_OVERRIDE=None
REVIEWER_CODE='DB'  # replace if required
PUBLIC_LINK=True
RANDOM_SEED=760042
REQUIRE_ALL_EXPOSED_PAGES=True
PAGE_ALLOWLIST=None

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook76_ring2_human_review'
OUT=OUTPUT_ROOT/RUN;INPUTS=OUT/'inputs';BLIND=OUT/'blind_review';SEALED=OUT/'sealed';RESP=OUT/'responses';TABLES=OUT/'tables';VISUAL=OUT/'visuals';AUDIT=OUT/'audit';UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,BLIND,SEALED,RESP,TABLES,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def object_hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp');t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str));t.replace(path)
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

NB75=Path(SOURCE_NB75_RUN_OVERRIDE) if SOURCE_NB75_RUN_OVERRIDE else latest(NB75_ROOT,['tables/results_by_page.csv','tables/results_by_component.csv','pages','prediction_lock/prediction_lock.json'])
NB75A=Path(SOURCE_NB75A_RUN_OVERRIDE) if SOURCE_NB75A_RUN_OVERRIDE else latest(NB75A_ROOT,['tables/page_role_resolution.csv','tables/collar_exposed_arm_summary.csv','notebook75a_verdict.json'])
require(NB75 is not None,'No complete Notebook 75 run found.')
require(NB75A is not None,'No complete Notebook 75A run found.')
require(REVIEWER_CODE and REVIEWER_CODE!='REPLACE_ME','Set REVIEWER_CODE.')
print('Notebook 75:',NB75);print('Notebook 75A:',NB75A);print('Notebook 76:',OUT)

## 2. Resolve the 51 collar-exposed pages and create the concealed review key

In [ ]:
results=pd.read_csv(NB75/'tables'/'results_by_page.csv')
components=pd.read_csv(NB75/'tables'/'results_by_component.csv')
roles=pd.read_csv(NB75A/'tables'/'page_role_resolution.csv')
exposed_roles=roles[roles.c2_restored_pixels>0].copy()
if PAGE_ALLOWLIST is not None: exposed_roles=exposed_roles[exposed_roles.page_key.isin(set(PAGE_ALLOWLIST))]
require(len(exposed_roles)>0,'No collar-exposed pages selected.')
# C1 truth-side page metrics are stored but sealed from initial reviewer display.
c1=results[(results.population=='PAIRED_DEVELOPMENT')&(results.arm=='C1_PRESERVE_RING2')].copy()
roster=exposed_roles[['page_key','page','source_group']].merge(c1[['page_key','true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components','restored_pixels']],on='page_key',validate='one_to_one')
roster=roster.sort_values(['source_group','page_key']).reset_index(drop=True)
require(len(roster)==exposed_roles.page_key.nunique(),'Roster resolution mismatch.')
# Deterministic random page order and an independently random A/B side assignment.
rng=np.random.default_rng(RANDOM_SEED);order=rng.permutation(len(roster));roster=roster.iloc[order].reset_index(drop=True);roster['review_order']=np.arange(1,len(roster)+1)
secret=secrets.token_bytes(32);secret_path=SEALED/'review_hmac_secret.bin';secret_path.write_bytes(secret)
key_rows=[];public_rows=[]
for _,r in roster.iterrows():
    pdir=NB75/'pages'/r.page_key;c0=pdir/'C0_INCUMBENT.png';c1p=pdir/'C1_PRESERVE_RING2.png'
    require(c0.exists() and c1p.exists(),f'{r.page_key}: missing C0/C1 page images')
    sides=['C0_INCUMBENT','C1_PRESERVE_RING2'];rng.shuffle(sides);mapping={'A':sides[0],'B':sides[1]}
    commit=hmac.new(secret,json.dumps({'page_key':r.page_key,'mapping':mapping},sort_keys=True).encode(),hashlib.sha256).hexdigest()
    key_rows.append({**r.to_dict(),'side_A':mapping['A'],'side_B':mapping['B'],'assignment_commitment':commit,'c0_sha256':sha256_file(c0),'c1_sha256':sha256_file(c1p)})
    public_rows.append({'review_order':int(r.review_order),'page_key':r.page_key,'source_group_blinded':'SET_'+hashlib.sha256(str(r.source_group).encode()).hexdigest()[:8],'assignment_commitment':commit})
key=pd.DataFrame(key_rows);public=pd.DataFrame(public_rows)
key.to_csv(SEALED/'review_key.csv',index=False);public.to_csv(BLIND/'review_queue.csv',index=False)
# Truth-side metrics go into a sealed table so the UI cannot display them accidentally.
roster[['page_key','true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components','restored_pixels']].to_csv(SEALED/'truth_side_page_metrics.csv',index=False)
print('Required blind-review pages:',len(public))

## 3. Build blind A/B images

The source page and clean truth are deliberately omitted from the initial comparison. A and B are displayed at identical scale and contrast.

In [ ]:
def compose_pair(a_path,b_path,label_a='A',label_b='B'):
    a=Image.open(a_path).convert('L');b=Image.open(b_path).convert('L')
    require(a.size==b.size,f'Image-size mismatch: {a_path} / {b_path}')
    # Downscale only for review portability while preserving identical scale.
    max_h=1800;scale=min(1.0,max_h/a.height);size=(max(1,int(a.width*scale)),max(1,int(a.height*scale)))
    a=a.resize(size,Image.Resampling.LANCZOS);b=b.resize(size,Image.Resampling.LANCZOS)
    banner=55;gap=18;canvas=Image.new('RGB',(size[0]*2+gap,size[1]+banner),'white');canvas.paste(a.convert('RGB'),(0,banner));canvas.paste(b.convert('RGB'),(size[0]+gap,banner));d=ImageDraw.Draw(canvas);d.text((15,15),label_a,fill='black');d.text((size[0]+gap+15,15),label_b,fill='black');return canvas

for _,r in key.iterrows():
    pdir=NB75/'pages'/r.page_key
    paths={'C0_INCUMBENT':pdir/'C0_INCUMBENT.png','C1_PRESERVE_RING2':pdir/'C1_PRESERVE_RING2.png'}
    im=compose_pair(paths[r.side_A],paths[r.side_B]);im.save(BLIND/f'{int(r.review_order):03d}_{r.page_key}_A_B.png')
print('Blind pairs:',len(list(BLIND.glob('*_A_B.png'))))

## 4. Persistent review store and typed queue helpers

In [ ]:
RESPONSE_PATH=RESP/'human_reviews.csv';HISTORY_PATH=RESP/'review_history.csv';LOCK_PATH=RESP/'human_review_lock.json'
FIELDS=['review_order','page_key','reviewer_code','submitted_utc','preferred_side','playability_A','playability_B','useful_print_restored','annotation_returned','annotation_interference','would_choose_candidate','confidence','notes']

def read_responses():
    if not RESPONSE_PATH.exists():return pd.DataFrame(columns=FIELDS)
    q=pd.read_csv(RESPONSE_PATH,dtype=str).fillna('')
    if len(q):q['review_order']=pd.to_numeric(q['review_order'],errors='raise').astype(int);q=q.sort_values(['review_order','page_key'],kind='stable')
    return q

def append_review(row):
    require(not LOCK_PATH.exists(),'Human-review lock is already closed.')
    q=read_responses();old=q[q.page_key.eq(row['page_key'])]
    if len(old):
        hist=pd.read_csv(HISTORY_PATH,dtype=str).fillna('') if HISTORY_PATH.exists() else pd.DataFrame(columns=FIELDS+['superseded_utc'])
        o=old.iloc[-1].to_dict();o['superseded_utc']=datetime.now(timezone.utc).isoformat();hist=pd.concat([hist,pd.DataFrame([o])],ignore_index=True);hist.to_csv(HISTORY_PATH,index=False);q=q[~q.page_key.eq(row['page_key'])]
    q=pd.concat([q,pd.DataFrame([row])],ignore_index=True);q['review_order']=pd.to_numeric(q.review_order,errors='raise').astype(int);q=q.sort_values(['review_order','page_key'],kind='stable');q.to_csv(RESPONSE_PATH,index=False)

def pending_queue():
    done=set(read_responses().page_key);q=public[~public.page_key.isin(done)].copy();q['review_order']=pd.to_numeric(q.review_order,errors='raise').astype(int);return q.sort_values(['review_order','page_key'])

def load_pair_for_page(page_key):
    r=public[public.page_key.eq(page_key)].iloc[0];return str(BLIND/f'{int(r.review_order):03d}_{page_key}_A_B.png')

## 5. Gradio whole-page review

**Interpret the questions visually.** The initial judgement is about the page as a musical object, not compliance with the historical pixel utility.

The key preference question explicitly allows `SAME / TRADE-OFF`.

In [ ]:
def ui_state():
    q=pending_queue();done=len(public)-len(q)
    if len(q)==0:return None,f'Complete: {done}/{len(public)}'
    r=q.iloc[0];return r.page_key,f'Completed {done}/{len(public)}. Current review order {int(r.review_order)}.'

def load_next():
    p,msg=ui_state()
    if p is None:return None,None,msg
    return p,load_pair_for_page(p),msg

def save_review(page_key,preferred,pa,pb,print_restore,ann_return,ann_interf,choose,confidence,notes):
    require(page_key is not None,'No page selected.')
    required={'preference':preferred,'playability A':pa,'playability B':pb,'print restoration':print_restore,'annotation return':ann_return,'annotation interference':ann_interf,'candidate choice':choose,'confidence':confidence}
    missing=[k for k,v in required.items() if not str(v).strip()]
    if missing:raise gr.Error('Complete: '+', '.join(missing))
    r=public[public.page_key.eq(page_key)].iloc[0]
    append_review({'review_order':int(r.review_order),'page_key':page_key,'reviewer_code':REVIEWER_CODE,'submitted_utc':datetime.now(timezone.utc).isoformat(),'preferred_side':preferred,'playability_A':pa,'playability_B':pb,'useful_print_restored':print_restore,'annotation_returned':ann_return,'annotation_interference':ann_interf,'would_choose_candidate':choose,'confidence':confidence,'notes':notes})
    nxt,img,msg=load_next();return nxt,img,'Saved. '+msg

with gr.Blocks(title='Notebook 76 Ring-2 Review') as app:
    gr.Markdown('# Notebook 76 — Blind C0 vs C1 Ring-2 human review')
    page=gr.Textbox(label='Blind page key',interactive=False);pair=gr.Image(type='filepath',label='Randomised A / B whole-page comparison',height=760);status=gr.Textbox(interactive=False,label='Progress')
    preferred=gr.Radio(['A','B','SAME / TRADE-OFF'],label='Which page would you rather read/perform from overall?')
    with gr.Row():
        pa=gr.Radio(['YES','YES_WITH_DIFFICULTY','NO','UNCERTAIN'],label='A: playable/readable?');pb=gr.Radio(['YES','YES_WITH_DIFFICULTY','NO','UNCERTAIN'],label='B: playable/readable?')
    print_restore=gr.Radio(['NONE','MINOR','MATERIAL','MAJOR','UNCERTAIN'],label='Does one version visibly restore useful printed notation compared with the other?')
    ann_return=gr.Radio(['NONE','MINOR','MATERIAL','OBJECTIONABLE','UNCERTAIN'],label='Does one version visibly retain/restore more annotation?')
    ann_interf=gr.Radio(['NO','SLIGHTLY','YES','UNCERTAIN'],label='Does the extra annotation, if any, interfere with reading the music?')
    choose=gr.Radio(['YES','SAME / INDIFFERENT','NO','UNCERTAIN'],label='If the candidate side were revealed, would you choose the page with the Ring-2 restoration for performance? Judge on visual trade-off, not pixel purity.')
    confidence=gr.Radio(['HIGH','MEDIUM','LOW'],label='Review confidence');notes=gr.Textbox(label='Notes',lines=3)
    save=gr.Button('Save and load next',variant='primary')
    app.load(load_next,outputs=[page,pair,status]);save.click(save_review,[page,preferred,pa,pb,print_restore,ann_return,ann_interf,choose,confidence,notes],[page,pair,status])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 6. Close the human-review lock only after the required queue is complete

In [ ]:
reviews=read_responses();missing=sorted(set(public.page_key)-set(reviews.page_key))
if REQUIRE_ALL_EXPOSED_PAGES:require(not missing,f'Review incomplete: {len(missing)} page(s) remain.')
require(len(reviews)>0,'No reviews recorded.')
lock_payload={'reviewer_code':REVIEWER_CODE,'reviewed_pages':len(reviews),'required_pages':len(public),'response_sha256':sha256_file(RESPONSE_PATH),'public_queue_sha256':sha256_file(BLIND/'review_queue.csv'),'assignment_key_sha256':sha256_file(SEALED/'review_key.csv'),'source_nb75':str(NB75),'source_nb75a':str(NB75A)}
review_lock={'lock_id':object_hash(lock_payload),'created_utc':datetime.now(timezone.utc).isoformat(),'state':'CLOSED_BEFORE_REVEAL','payload':lock_payload}
atomic_json(LOCK_PATH,review_lock);print(json.dumps(review_lock,indent=2))

## 7. Reveal A/B identities and decode the human review

In [ ]:
require(LOCK_PATH.exists(),'Close the human-review lock first.')
reviews=read_responses();key=pd.read_csv(SEALED/'review_key.csv')
decoded=reviews.merge(key[['page_key','side_A','side_B','source_group','true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components','restored_pixels']],on='page_key',validate='one_to_one')

def actual_pref(r):
    if r.preferred_side=='SAME / TRADE-OFF':return 'SAME_OR_TRADEOFF'
    return r.side_A if r.preferred_side=='A' else r.side_B

def candidate_side(r):return 'A' if r.side_A=='C1_PRESERVE_RING2' else 'B'
def incumbent_side(r):return 'A' if r.side_A=='C0_INCUMBENT' else 'B'
def side_value(r,prefix,side):return r[f'{prefix}_{side}']

decoded['preferred_actual']=decoded.apply(actual_pref,axis=1);decoded['candidate_side']=decoded.apply(candidate_side,axis=1);decoded['incumbent_side']=decoded.apply(incumbent_side,axis=1)
decoded['candidate_playability']=decoded.apply(lambda r:side_value(r,'playability',r.candidate_side),axis=1);decoded['incumbent_playability']=decoded.apply(lambda r:side_value(r,'playability',r.incumbent_side),axis=1)
decoded['candidate_preferred']=decoded.preferred_actual.eq('C1_PRESERVE_RING2');decoded['incumbent_preferred']=decoded.preferred_actual.eq('C0_INCUMBENT');decoded['tradeoff_tie']=decoded.preferred_actual.eq('SAME_OR_TRADEOFF')
decoded.to_csv(TABLES/'decoded_page_reviews.csv',index=False)
print(decoded[['page_key','candidate_preferred','incumbent_preferred','tradeoff_tie','candidate_playability','incumbent_playability','annotation_interference','would_choose_candidate','catastrophic_components']].head())

## 8. Human preference, readability and mathematical-failure summaries

In [ ]:
summary={
 'reviewed_pages':len(decoded),
 'candidate_preferred_pages':int(decoded.candidate_preferred.sum()),
 'incumbent_preferred_pages':int(decoded.incumbent_preferred.sum()),
 'same_or_tradeoff_pages':int(decoded.tradeoff_tie.sum()),
 'candidate_playable_yes':int(decoded.candidate_playability.eq('YES').sum()),
 'incumbent_playable_yes':int(decoded.incumbent_playability.eq('YES').sum()),
 'candidate_choice_yes':int(decoded.would_choose_candidate.eq('YES').sum()),
 'candidate_choice_no':int(decoded.would_choose_candidate.eq('NO').sum()),
 'annotation_interference_yes':int(decoded.annotation_interference.eq('YES').sum()),
 'annotation_interference_slightly':int(decoded.annotation_interference.eq('SLIGHTLY').sum()),
 'annotation_interference_no':int(decoded.annotation_interference.eq('NO').sum()),
 'mathematical_catastrophe_pages':int((decoded.catastrophic_components>0).sum()),
}
# Catastrophic pages and false-heavy pages are evaluated after reveal, using already locked human responses.
cat=decoded[decoded.catastrophic_components>0].copy();false_heavy=decoded.sort_values(['annotation_retained','true_print_recovered'],ascending=[False,True]).head(min(12,len(decoded))).copy()
cat.to_csv(TABLES/'mathematical_catastrophe_human_reviews.csv',index=False);false_heavy.to_csv(TABLES/'false_heavy_human_reviews.csv',index=False)
summary['catastrophe_pages_candidate_preferred']=int(cat.candidate_preferred.sum());summary['catastrophe_pages_annotation_interference_yes']=int(cat.annotation_interference.eq('YES').sum());summary['catastrophe_pages_candidate_choice_no']=int(cat.would_choose_candidate.eq('NO').sum())
atomic_json(TABLES/'human_review_summary.json',summary)
# Source stability in human terms.
by_group=decoded.groupby('source_group',as_index=False).agg(pages=('page_key','size'),candidate_preferred=('candidate_preferred','sum'),incumbent_preferred=('incumbent_preferred','sum'),tradeoff_tie=('tradeoff_tie','sum'),candidate_choice_yes=('would_choose_candidate',lambda x:int((x=='YES').sum())),annotation_interference_yes=('annotation_interference',lambda x:int((x=='YES').sum())))
by_group.to_csv(TABLES/'human_review_by_source_group.csv',index=False)
print(json.dumps(summary,indent=2));print(by_group.to_string(index=False))

## 9. Visual review of the mathematically catastrophic / false-heavy cases after reveal

These diagnostics are intentionally post-lock. The reviewer can now inspect the C0/C1 identity and truth-side labels without contaminating the initial preference judgement.

In [ ]:
def contact(rows,name,max_rows=12):
    rows=rows.head(max_rows);fig,axes=plt.subplots(len(rows),4,figsize=(18,max(5,5.2*len(rows))),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        d=NB75/'pages'/r.page_key
        ims=[Image.open(d/'observed.png').convert('L'),Image.open(d/'C0_INCUMBENT.png').convert('L'),Image.open(d/'C1_PRESERVE_RING2.png').convert('L'),Image.open(d/'C1_PRESERVE_RING2_truth_overlay_posthoc.png').convert('RGB') if (d/'C1_PRESERVE_RING2_truth_overlay_posthoc.png').exists() else Image.open(d/'C1_PRESERVE_RING2.png').convert('RGB')]
        titles=['observed','C0 incumbent','C1 Ring2','post-hoc truth overlay']
        for j,(im,t) in enumerate(zip(ims,titles)):
            axes[i,j].imshow(np.asarray(im),cmap='gray' if im.mode=='L' else None);axes[i,j].set_title(f'{r.page_key} {t}\nP={int(r.true_print_recovered)} A={int(r.annotation_retained)} human={r.preferred_actual}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=130,bbox_inches='tight');plt.close(fig)
if len(cat):contact(cat,'catastrophic_cases_post_reveal',12)
contact(false_heavy,'false_heavy_cases_post_reveal',12)

## 10. Preference charts and practical decision evidence

In [ ]:
sns.set_theme(style='whitegrid')
# Overall preference.
labels=['C1 Ring2','C0 incumbent','Same/trade-off'];vals=[summary['candidate_preferred_pages'],summary['incumbent_preferred_pages'],summary['same_or_tradeoff_pages']]
plt.figure(figsize=(8,5));sns.barplot(x=labels,y=vals);plt.ylabel('Pages');plt.title('Human whole-page preference after blinded A/B review');plt.tight_layout();plt.savefig(VISUAL/'overall_human_preference.png',dpi=180);plt.close()
# Human choice versus mathematical annotation cost.
plt.figure(figsize=(9,7));sns.scatterplot(data=decoded,x='true_print_recovered',y='annotation_retained',hue='would_choose_candidate',style='annotation_interference',size='restored_pixels',sizes=(25,260),alpha=.85);plt.title('Would the reviewer choose C1 despite returned annotation?');plt.tight_layout();plt.savefig(VISUAL/'human_choice_vs_truth_tradeoff.png',dpi=180);plt.close()
# Source group preference ratio.
g=by_group.copy();g['candidate_preference_rate']=g.candidate_preferred/g.pages;g=g.sort_values('candidate_preference_rate')
plt.figure(figsize=(10,max(6,.45*len(g))));plt.barh(g.source_group,g.candidate_preference_rate);plt.xlim(0,1);plt.xlabel('C1 preferred / reviewed pages');plt.title('Ring-2 preference by source group');plt.tight_layout();plt.savefig(VISUAL/'candidate_preference_by_source_group.png',dpi=180);plt.close()

## 11. Verdict record and final review bundle

In [ ]:
# No automatic production verdict is inferred from preference alone.
# The record distinguishes practical desirability from automatic-safety evidence.
cat_visual_objection = bool(len(cat) and ((cat.annotation_interference=='YES') | (cat.would_choose_candidate=='NO')).any())
record={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'version':'notebook76-ring2-human-review',
 'source_notebook75':str(NB75),'source_notebook75a':str(NB75A),
 'human_review_lock_id':json.loads(LOCK_PATH.read_text())['lock_id'],
 'review_summary':summary,
 'catastrophic_cases_visibly_objectionable':cat_visual_objection,
 'interpretation':(
   'C1 whole-page preference and playability are human outcomes, not substitutes for truth-side safety metrics. '
   'If mathematically false-heavy/catastrophic pages are consistently visually acceptable and C1 is preferred, '
   'future policy design may treat pixel safety as an exception/veto problem rather than rejecting Ring 2 wholesale. '
   'If such cases are visibly objectionable or interfere with reading, an automatic veto/selector remains justified.'
 ),
 'operator_changed':False,'promotion_authority':False,
}
atomic_json(OUT/'notebook76_verdict.json',record)
lines=['NOTEBOOK 76 — RING-2 HUMAN PREFERENCE AND OBJECTIONABILITY REVIEW','',f'Notebook 75: {NB75}',f'Notebook 75A: {NB75A}',f'Reviewed pages: {len(decoded)}',f'C1 preferred: {summary["candidate_preferred_pages"]}',f'C0 preferred: {summary["incumbent_preferred_pages"]}',f'Same/trade-off: {summary["same_or_tradeoff_pages"]}',f'Would choose C1: {summary["candidate_choice_yes"]}',f'Would not choose C1: {summary["candidate_choice_no"]}',f'Annotation interferes with reading: {summary["annotation_interference_yes"]}',f'Mathematical catastrophe pages: {summary["mathematical_catastrophe_pages"]}',f'Catastrophe visually objectionable under review rule: {cat_visual_objection}','','No automatic production decision is made. Human preference is evaluated alongside, not instead of, the locked truth-side evidence.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook76_verdict.json',LOCK_PATH,TABLES/'decoded_page_reviews.csv',TABLES/'human_review_summary.json',TABLES/'human_review_by_source_group.csv',TABLES/'mathematical_catastrophe_human_reviews.csv',TABLES/'false_heavy_human_reviews.csv',VISUAL/'overall_human_preference.png',VISUAL/'human_choice_vs_truth_tradeoff.png',VISUAL/'candidate_preference_by_source_group.png']
for p in required:require(p.exists(),f'Missing {p}');shutil.copy2(p,UPLOAD/p.name)
for p in [VISUAL/'catastrophic_cases_post_reveal.png',VISUAL/'false_heavy_cases_post_reveal.png']:
    if p.exists():shutil.copy2(p,UPLOAD/p.name)
manifest=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'README.txt').write_text('Notebook 76 blinded Ring-2 whole-page review. Start with executive_summary.txt, decoded_page_reviews.csv, human_review_summary.json and the preference/false-heavy visuals.\n')
zip_path=UPLOAD/'notebook76_ring2_human_review_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p!=zip_path:z.write(p,p.name)
print('Bundle:',zip_path)

## How to interpret Notebook 76

- A page may legitimately prefer C1 even when some annotation returns. The primary practical endpoint is readability/playability of the whole page.
- A mathematically false-heavy page is not automatically a visually bad page. Notebook 76 tests that directly after the blinded review is locked.
- Conversely, a page with favourable aggregate pixel trade-off may still be visually objectionable if returned annotation obscures notation or distracts the reader.
- If C1 is preferred broadly and the mathematically adverse cases are visually acceptable, the next experiment should test a **small objectionability/exception veto**, not automatically revert to an ultra-conservative pixel rule.
- If adverse cases are visibly objectionable or interfere with reading, then a selective Ring-2 ownership/veto model remains necessary.
- `v8_8_operator.py` remains unchanged throughout this notebook.